# NYC Taxi — CatBoost GPU + KNN features + Linear Regression comparison

Этот ноутбук собирает в одном месте финальный ML-блок для **прогноза стоимости поездки (`total_amount`)**:

- вместо старого `HistGradientBoostingRegressor` используется **CatBoostRegressor на GPU**;
- к бустингу добавлены **KNN target-stat features**: локальная средняя / медианная / разброс цены похожих поездок;
- есть честное сравнение с **Linear Regression baseline** из предыдущего ноутбука;
- все ключевые графики сделаны через **Plotly** в презентационном стиле: чёрный фон, белые подписи, цветные столбцы;
- дефолтный режим настроен на быстрый запуск на Kaggle GPU примерно до минуты.

> Важно: ноутбук не использует прямые денежные компоненты `fare_amount`, `tip_amount`, `tolls_amount`, `airport_fee`, `congestion_surcharge` как признаки, потому что они раскрывают `total_amount`.


In [1]:
from __future__ import annotations

import gc
import importlib
import json
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


def ensure_package(import_name: str, pip_name: str | None = None):
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        return importlib.import_module(import_name)


pa = ensure_package("pyarrow")
pq = ensure_package("pyarrow.parquet", "pyarrow")
pads = ensure_package("pyarrow.dataset", "pyarrow")

catboost_module = ensure_package("catboost")
from catboost import CatBoostRegressor, Pool

plotly = ensure_package("plotly")
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

try:
    import kaleido
    KALEIDO_AVAILABLE = True
except Exception as exc:
    KALEIDO_AVAILABLE = False


In [2]:
RUN_MODE = "presentation_quality"

DATA_FILE_NAME = "my_clean_3_with_weather.parquet"
TARGET = "total_amount"
DATETIME_COL = "tpep_pickup_datetime"
DROPOFF_DATETIME_COL = "tpep_dropoff_datetime"

USE_DURATION_FEATURES = True

ALLOW_CPU_FALLBACK = False
USE_ALL_GPUS = False
EXPORT_PLOTLY_HTML = True
EXPORT_PLOTLY_PNG = True

VALID_START = pd.Timestamp("2024-10-01")
DATE_MIN = pd.Timestamp("2023-01-01")
DATE_MAX = pd.Timestamp("2025-01-01")

RUN_MODES = {
    "smoke_test": {
        "max_rows": 50_000,
        "valid_size": 0.20,
        "cat_iterations": 140,
        "cat_depth": 4,
        "cat_learning_rate": 0.12,
        "early_stopping_rounds": 20,
        "knn_neighbors": 15,
        "knn_reference_rows": 6_000,
        "knn_oof_folds": 3,
        "knn_query_chunk_size": 20_000,
        "target_stat_folds": 2,
        "plot_sample_rows": 15_000,
        "linear_alpha": 2.0,
    },
    "one_minute_gpu": {
        "max_rows": 120_000,
        "valid_size": 0.20,
        "cat_iterations": 300,
        "cat_depth": 5,
        "cat_learning_rate": 0.09,
        "early_stopping_rounds": 30,
        "knn_neighbors": 25,
        "knn_reference_rows": 15_000,
        "knn_oof_folds": 3,
        "knn_query_chunk_size": 25_000,
        "target_stat_folds": 2,
        "plot_sample_rows": 30_000,
        "linear_alpha": 2.0,
    },
    "presentation_quality": {
        "max_rows": 500_000,
        "valid_size": 0.20,
        "cat_iterations": 1200,
        "cat_depth": 7,
        "cat_learning_rate": 0.045,
        "early_stopping_rounds": 90,
        "knn_neighbors": 55,
        "knn_reference_rows": 90_000,
        "knn_oof_folds": 3,
        "knn_query_chunk_size": 45_000,
        "target_stat_folds": 4,
        "plot_sample_rows": 70_000,
        "linear_alpha": 2.0,
    },
}

CFG = RUN_MODES[RUN_MODE].copy()

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data/outputs_catboost_knn_plotly")
FIG_DIR = OUTPUT_DIR / "plotly_visuals"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
def find_data_file(file_name: str) -> Path:
    direct_candidates = [
        Path(file_name),
        Path.cwd() / file_name,
        Path.cwd() / "data" / file_name,
        Path.cwd() / "datasets" / file_name,
        Path("/kaggle/working") / file_name,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    search_roots = [Path("/kaggle/input"), Path.cwd(), Path.cwd().parent]
    found: list[Path] = []
    for root in search_roots:
        if root.exists():
            try:
                found.extend(root.rglob(file_name))
            except Exception:
                pass
    if found:
        return sorted(found, key=lambda p: len(str(p)))[0]

    checked = "\n".join(str(p) for p in direct_candidates + search_roots)
    raise FileNotFoundError(
        f"Не найден {file_name}. Добавь датасет в Kaggle Input или положи файл рядом с ноутбуком.\nChecked:\n{checked}"
    )


def get_nvidia_gpus() -> list[str]:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            check=True,
            capture_output=True,
            text=True,
        )
        return [line.strip() for line in result.stdout.splitlines() if line.strip()]
    except Exception as exc:
        return []


DATA_PATH = find_data_file(DATA_FILE_NAME)
GPU_LIST = get_nvidia_gpus()

if GPU_LIST:
    CATBOOST_TASK_TYPE = "GPU"
    CATBOOST_DEVICES = ":".join(str(i) for i in range(len(GPU_LIST))) if USE_ALL_GPUS else "0"
else:
    if not ALLOW_CPU_FALLBACK:
        raise RuntimeError("GPU не найден. В Kaggle включи Accelerator -> GPU и перезапусти session.")
    CATBOOST_TASK_TYPE = "CPU"
    CATBOOST_DEVICES = None


RuntimeError: GPU не найден. В Kaggle включи Accelerator -> GPU и перезапусти session.

In [ ]:
LEAKAGE_MONEY_COLS = {
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
    "total_amount",
}

BASE_WANTED_COLS = [
    TARGET,
    DATETIME_COL,
    DROPOFF_DATETIME_COL,
    "duration_min",
    "trip_duration_min",
    "trip_distance",
    "pickup_hour",
    "distance_group",
    "passenger_count",
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type",
    "PULocationID",
    "DOLocationID",
    "PU_Borough",
    "DO_Borough",
    "PU_Zone",
    "DO_Zone",
    "PU_zone",
    "DO_zone",
    "PU_zone_name",
    "DO_zone_name",
    "pickup_zone",
    "dropoff_zone",
    "PU_lon",
    "PU_lat",
    "DO_lon",
    "DO_lat",
    "temperature",
    "precipitation",
    "snowfall",
    "weather_code",
]


def parquet_schema_columns(path: Path) -> list[str]:
    if path.is_dir():
        dataset = pads.dataset(path, format="parquet")
        return list(dataset.schema.names)
    return list(pq.read_schema(path).names)


def read_parquet_limited(path: Path, columns: list[str], max_rows: int | None) -> pd.DataFrame:
    """Fast read: in one-minute mode, read first N rows from Parquet instead of full 7M+ rows."""
    if max_rows is None:
        return pd.read_parquet(path, columns=columns)

    if path.is_dir():
        dataset = pads.dataset(path, format="parquet")
        table = dataset.head(max_rows, columns=columns)
        return table.to_pandas()

    parquet_file = pq.ParquetFile(path)
    batches = []
    rows_read = 0
    for batch in parquet_file.iter_batches(batch_size=200_000, columns=columns):
        need = max_rows - rows_read
        if need <= 0:
            break
        if batch.num_rows > need:
            batch = batch.slice(0, need)
        batches.append(batch)
        rows_read += batch.num_rows
        if rows_read >= max_rows:
            break
    if not batches:
        return pd.DataFrame(columns=columns)
    table = pa.Table.from_batches(batches)
    return table.to_pandas()


schema_cols = parquet_schema_columns(DATA_PATH)
schema_set = set(schema_cols)

read_cols = [col for col in BASE_WANTED_COLS if col in schema_set]
if TARGET not in read_cols:
    raise ValueError(f"Target column {TARGET!r} not found in data.")
if DATETIME_COL not in read_cols:
    raise ValueError(f"Datetime column {DATETIME_COL!r} not found in data.")

load_start = time.time()
df = read_parquet_limited(DATA_PATH, read_cols, CFG["max_rows"])


if "duration_min" not in df.columns and "trip_duration_min" in df.columns:
    df = df.rename(columns={"trip_duration_min": "duration_min"})


df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], errors="coerce")
if "duration_min" not in df.columns and DROPOFF_DATETIME_COL in df.columns:
    df[DROPOFF_DATETIME_COL] = pd.to_datetime(df[DROPOFF_DATETIME_COL], errors="coerce")
    df["duration_min"] = (df[DROPOFF_DATETIME_COL] - df[DATETIME_COL]).dt.total_seconds() / 60


for col in [TARGET, "trip_distance", "duration_min", "passenger_count", "PU_lon", "PU_lat", "DO_lon", "DO_lat", "temperature", "precipitation", "snowfall"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


filters = df[DATETIME_COL].ge(DATE_MIN) & df[DATETIME_COL].lt(DATE_MAX)
filters &= df[TARGET].between(0, 500)
if "trip_distance" in df.columns:
    filters &= df["trip_distance"].between(0, 150)
if "duration_min" in df.columns:
    filters &= df["duration_min"].between(0.1, 300)
if "passenger_count" in df.columns:
    df["passenger_count"] = df["passenger_count"].fillna(1).clip(1, 6)

df = df.loc[filters].copy()
df = df.sort_values(DATETIME_COL).reset_index(drop=True)

if len(df) < 5_000:
    raise ValueError(f"После фильтрации осталось слишком мало строк: {len(df)}")

display(df[TARGET].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame().T)


In [ ]:
def ensure_numeric_col(frame: pd.DataFrame, col: str, default: float = 0.0) -> None:
    if col not in frame.columns:
        frame[col] = default
    frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(default)


def ensure_str_col(frame: pd.DataFrame, col: str, default: str = "unknown") -> None:
    if col not in frame.columns:
        frame[col] = default
    frame[col] = frame[col].astype("string").fillna(default).replace("<NA>", default).astype(str)


for col in ["trip_distance", "duration_min", "passenger_count", "PU_lon", "PU_lat", "DO_lon", "DO_lat", "temperature", "precipitation", "snowfall"]:
    ensure_numeric_col(df, col, 0.0)

for col in ["VendorID", "RatecodeID", "store_and_fwd_flag", "payment_type", "PULocationID", "DOLocationID", "PU_Borough", "DO_Borough", "weather_code"]:
    ensure_str_col(df, col)


def normalize_id_string(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    out = numeric.round().astype("Int64").astype("string")
    out = out.fillna(series.astype("string").fillna("unknown"))
    return out.fillna("unknown").astype(str)

for id_col in ["VendorID", "RatecodeID", "payment_type", "PULocationID", "DOLocationID"]:
    df[id_col] = normalize_id_string(df[id_col])

if "pickup_hour" not in df.columns:
    df["pickup_hour"] = df[DATETIME_COL].dt.hour
else:
    df["pickup_hour"] = pd.to_numeric(df["pickup_hour"], errors="coerce").fillna(df[DATETIME_COL].dt.hour)

if "distance_group" not in df.columns:
    df["distance_group"] = pd.cut(
        df["trip_distance"].clip(lower=0),
        bins=[-0.01, 1, 3, 7, 15, np.inf],
        labels=["very_short", "short", "medium", "long", "very_long"],
    ).astype(str)
else:
    ensure_str_col(df, "distance_group")


df["pickup_hour"] = df["pickup_hour"].astype("int16")
df["pickup_dayofweek"] = df[DATETIME_COL].dt.dayofweek.astype("int16")
df["pickup_month"] = df[DATETIME_COL].dt.month.astype("int16")
df["pickup_day"] = df[DATETIME_COL].dt.day.astype("int16")
df["pickup_weekofyear"] = df[DATETIME_COL].dt.isocalendar().week.astype("int16")
df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype("int8")
df["is_rush_hour"] = (
    (df["pickup_dayofweek"] < 5)
    & (((df["pickup_hour"] >= 7) & (df["pickup_hour"] <= 10)) | ((df["pickup_hour"] >= 16) & (df["pickup_hour"] <= 20)))
).astype("int8")
df["is_night_tariff"] = ((df["pickup_hour"] >= 20) | (df["pickup_hour"] < 6)).astype("int8")
df["hour_sin"] = np.sin(2 * np.pi * df["pickup_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["pickup_hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["pickup_dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["pickup_dayofweek"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["pickup_month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["pickup_month"] / 12)


df["trip_distance"] = df["trip_distance"].clip(lower=0)
df["duration_min"] = df["duration_min"].clip(lower=0.1)
df["log_trip_distance"] = np.log1p(df["trip_distance"])
df["log_duration_min"] = np.log1p(df["duration_min"])
df["speed_mph"] = (df["trip_distance"] / (df["duration_min"] / 60 + 1e-3)).clip(0, 80)


def haversine_miles(lon1, lat1, lon2, lat2):
    lon1 = np.radians(lon1.astype(float))
    lat1 = np.radians(lat1.astype(float))
    lon2 = np.radians(lon2.astype(float))
    lat2 = np.radians(lat2.astype(float))
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


df["lat_diff"] = (df["DO_lat"] - df["PU_lat"]).abs()
df["lon_diff"] = (df["DO_lon"] - df["PU_lon"]).abs()
df["gps_distance_miles"] = haversine_miles(df["PU_lon"], df["PU_lat"], df["DO_lon"], df["DO_lat"]).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 150)
df["distance_ratio"] = (df["trip_distance"] / (df["gps_distance_miles"] + 0.1)).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 20)


df["has_precipitation"] = (df["precipitation"] > 0).astype("int8")
df["has_snowfall"] = (df["snowfall"] > 0).astype("int8")
df["bad_weather"] = ((df["precipitation"] > 0) | (df["snowfall"] > 0)).astype("int8")
df["freezing_weather"] = (df["temperature"] <= 0).astype("int8")
df["precipitation_x_rush_hour"] = df["precipitation"] * df["is_rush_hour"]
df["snowfall_x_rush_hour"] = df["snowfall"] * df["is_rush_hour"]
df["temperature_x_precipitation"] = df["temperature"] * df["precipitation"]


airport_zone_ids = {"1", "132", "138"}
pu_loc = df["PULocationID"].astype(str)
do_loc = df["DOLocationID"].astype(str)
df["pickup_airport"] = pu_loc.isin(airport_zone_ids).astype("int8")
df["dropoff_airport"] = do_loc.isin(airport_zone_ids).astype("int8")
df["airport_trip"] = ((df["pickup_airport"] == 1) | (df["dropoff_airport"] == 1)).astype("int8")
df["same_zone"] = (pu_loc == do_loc).astype("int8")
df["same_borough"] = (df["PU_Borough"].astype(str) == df["DO_Borough"].astype(str)).astype("int8")
df["interborough_trip"] = (df["same_borough"] == 0).astype("int8")
df["route_id"] = pu_loc + "→" + do_loc
df["borough_pair"] = df["PU_Borough"].astype(str) + "→" + df["DO_Borough"].astype(str)


zone_label_candidates = ["PU_Zone", "PU_zone", "PU_zone_name", "pickup_zone"]
for col in zone_label_candidates:
    if col in df.columns:
        ensure_str_col(df, col)

display(df[[TARGET, "trip_distance", "duration_min", "pickup_hour", "airport_trip", "bad_weather", "route_id"]].head())


In [ ]:
df_model = df.sort_values(DATETIME_COL).reset_index(drop=True)

valid_mask = df_model[DATETIME_COL].ge(VALID_START)
valid_ratio = float(valid_mask.mean())
if valid_mask.sum() >= 5_000 and 0.05 <= valid_ratio <= 0.50:
    train_df = df_model.loc[~valid_mask].copy().reset_index(drop=True)
    valid_df = df_model.loc[valid_mask].copy().reset_index(drop=True)
    split_type = f"time split: validation from {VALID_START.date()}"
else:
    split_idx = int(len(df_model) * (1 - CFG["valid_size"]))
    train_df = df_model.iloc[:split_idx].copy().reset_index(drop=True)
    valid_df = df_model.iloc[split_idx:].copy().reset_index(drop=True)
    split_type = f"fallback chronological split: last {CFG['valid_size']:.0%} rows"


y_train = train_df[TARGET].astype("float32").reset_index(drop=True)
y_valid = valid_df[TARGET].astype("float32").reset_index(drop=True)


del df
gc.collect()


In [ ]:
from sklearn.model_selection import KFold

TARGET_ENCODING_SOURCE_COLS = [
    "PULocationID",
    "DOLocationID",
    "route_id",
    "RatecodeID",
    "borough_pair",
]
TARGET_ENCODING_SOURCE_COLS = [col for col in TARGET_ENCODING_SOURCE_COLS if col in train_df.columns]


def safe_feature_name(prefix: str, source_col: str, suffix: str) -> str:
    cleaned = source_col.replace("LocationID", "loc").replace("_id", "").replace("→", "to")
    return f"{prefix}_{cleaned}_{suffix}"


def build_smoothed_mapping(source: pd.Series, target: pd.Series, smooth: float, global_mean: float):
    tmp = pd.DataFrame({"key": source.astype("string").fillna("unknown"), "target": target.to_numpy(dtype="float32")})
    stats = tmp.groupby("key", observed=False)["target"].agg(["mean", "count"])
    smoothed = (stats["mean"] * stats["count"] + global_mean * smooth) / (stats["count"] + smooth)
    return smoothed, stats["count"]


def add_oof_target_statistics(
    train_frame: pd.DataFrame,
    valid_frame: pd.DataFrame,
    train_target: pd.Series,
    source_cols: list[str],
    n_splits: int = 2,
    smooth: float = 80.0,
):
    train_frame = train_frame.copy()
    valid_frame = valid_frame.copy()
    global_mean = float(train_target.mean())
    created_cols: list[str] = []

    n_splits = max(2, min(n_splits, len(train_frame)))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for source_col in source_cols:
        mean_col = safe_feature_name("te", source_col, "mean")
        count_col = safe_feature_name("te", source_col, "count_log")
        created_cols.extend([mean_col, count_col])

        train_frame[mean_col] = np.nan
        keys_train = train_frame[source_col].astype("string").fillna("unknown")
        keys_valid = valid_frame[source_col].astype("string").fillna("unknown")


        for fit_idx, hold_idx in kf.split(train_frame):
            mapping, _ = build_smoothed_mapping(
                keys_train.iloc[fit_idx],
                train_target.iloc[fit_idx],
                smooth=smooth,
                global_mean=float(train_target.iloc[fit_idx].mean()),
            )
            train_frame.loc[hold_idx, mean_col] = keys_train.iloc[hold_idx].map(mapping).fillna(global_mean).astype("float32").to_numpy()


        full_mapping, full_counts = build_smoothed_mapping(keys_train, train_target, smooth=smooth, global_mean=global_mean)
        valid_frame[mean_col] = keys_valid.map(full_mapping).fillna(global_mean).astype("float32").to_numpy()


        train_frame[count_col] = np.log1p(keys_train.map(full_counts).fillna(0)).astype("float32").to_numpy()
        valid_frame[count_col] = np.log1p(keys_valid.map(full_counts).fillna(0)).astype("float32").to_numpy()

    return train_frame, valid_frame, created_cols


stat_start = time.time()
train_df, valid_df, target_stat_cols = add_oof_target_statistics(
    train_df,
    valid_df,
    y_train,
    TARGET_ENCODING_SOURCE_COLS,
    n_splits=CFG["target_stat_folds"],
    smooth=80.0,
)
display(train_df[target_stat_cols].head())


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

BASE_KNN_NUMERIC_COLS = [
    "trip_distance",
    "log_trip_distance",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "gps_distance_miles",
    "distance_ratio",
    "PU_lon",
    "PU_lat",
    "DO_lon",
    "DO_lat",
    "lat_diff",
    "lon_diff",
    "pickup_airport",
    "dropoff_airport",
    "airport_trip",
    "same_zone",
    "same_borough",
    "temperature",
    "precipitation",
    "snowfall",
    "bad_weather",
    *target_stat_cols,
]
if USE_DURATION_FEATURES:
    BASE_KNN_NUMERIC_COLS += ["duration_min", "log_duration_min", "speed_mph"]

KNN_NUMERIC_COLS = [col for col in dict.fromkeys(BASE_KNN_NUMERIC_COLS) if col in train_df.columns]
KNN_OUTPUT_COLS = [
    "knn_mean_total_amount",
    "knn_median_total_amount",
    "knn_std_total_amount",
    "knn_min_total_amount",
    "knn_max_total_amount",
    "knn_p25_total_amount",
    "knn_p75_total_amount",
    "knn_mean_distance",
    "knn_min_distance",
]


def sample_reference_indices(n_rows: int, max_rows: int, seed: int) -> np.ndarray:
    if n_rows <= max_rows:
        return np.arange(n_rows)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n_rows, size=max_rows, replace=False))


def knn_stats_from_reference(
    reference_df: pd.DataFrame,
    reference_y: pd.Series,
    query_df: pd.DataFrame,
    feature_cols: list[str],
    n_neighbors: int,
    max_reference_rows: int,
    query_chunk_size: int,
    seed: int,
) -> pd.DataFrame:
    ref_idx = sample_reference_indices(len(reference_df), max_reference_rows, seed)
    ref_part = reference_df.iloc[ref_idx]
    ref_y = reference_y.iloc[ref_idx].to_numpy(dtype="float32")

    k = int(min(n_neighbors, len(ref_part)))
    if k < 2:
        global_mean = float(reference_y.mean())
        return pd.DataFrame({
            "knn_mean_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_median_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_std_total_amount": np.zeros(len(query_df), dtype="float32"),
            "knn_min_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_max_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_p25_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_p75_total_amount": np.full(len(query_df), global_mean, dtype="float32"),
            "knn_mean_distance": np.zeros(len(query_df), dtype="float32"),
            "knn_min_distance": np.zeros(len(query_df), dtype="float32"),
        })

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_ref = ref_part[feature_cols].to_numpy(dtype="float32", copy=True)
    X_ref = imputer.fit_transform(X_ref)
    X_ref = scaler.fit_transform(X_ref).astype("float32")

    nn = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="euclidean", n_jobs=-1)
    nn.fit(X_ref)

    chunks = []
    for start in range(0, len(query_df), query_chunk_size):
        end = min(start + query_chunk_size, len(query_df))
        X_q = query_df.iloc[start:end][feature_cols].to_numpy(dtype="float32", copy=True)
        X_q = imputer.transform(X_q)
        X_q = scaler.transform(X_q).astype("float32")

        distances, indices = nn.kneighbors(X_q, return_distance=True)
        neighbor_values = ref_y[indices]

        chunk_stats = pd.DataFrame({
            "knn_mean_total_amount": neighbor_values.mean(axis=1),
            "knn_median_total_amount": np.median(neighbor_values, axis=1),
            "knn_std_total_amount": neighbor_values.std(axis=1),
            "knn_min_total_amount": neighbor_values.min(axis=1),
            "knn_max_total_amount": neighbor_values.max(axis=1),
            "knn_p25_total_amount": np.percentile(neighbor_values, 25, axis=1),
            "knn_p75_total_amount": np.percentile(neighbor_values, 75, axis=1),
            "knn_mean_distance": distances.mean(axis=1),
            "knn_min_distance": distances.min(axis=1),
        })
        chunks.append(chunk_stats.astype("float32"))

    return pd.concat(chunks, axis=0, ignore_index=True)


def add_oof_knn_features(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, train_target: pd.Series):
    train_knn = pd.DataFrame(index=train_frame.index, columns=KNN_OUTPUT_COLS, dtype="float32")

    n_splits = max(2, min(CFG["knn_oof_folds"], len(train_frame)))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    for fold, (ref_idx, query_idx) in enumerate(kf.split(train_frame), start=1):
        fold_start = time.time()
        stats = knn_stats_from_reference(
            train_frame.iloc[ref_idx],
            train_target.iloc[ref_idx],
            train_frame.iloc[query_idx],
            feature_cols=KNN_NUMERIC_COLS,
            n_neighbors=CFG["knn_neighbors"],
            max_reference_rows=CFG["knn_reference_rows"],
            query_chunk_size=CFG["knn_query_chunk_size"],
            seed=RANDOM_STATE + fold,
        )
        train_knn.loc[query_idx, KNN_OUTPUT_COLS] = stats.to_numpy(dtype="float32")

    valid_knn = knn_stats_from_reference(
        train_frame,
        train_target,
        valid_frame,
        feature_cols=KNN_NUMERIC_COLS,
        n_neighbors=CFG["knn_neighbors"],
        max_reference_rows=CFG["knn_reference_rows"],
        query_chunk_size=CFG["knn_query_chunk_size"],
        seed=RANDOM_STATE + 999,
    )

    for col in KNN_OUTPUT_COLS:
        train_frame[col] = train_knn[col].astype("float32").fillna(float(train_target.mean())).to_numpy()
        valid_frame[col] = valid_knn[col].astype("float32").to_numpy()

    return train_frame, valid_frame


knn_start = time.time()
train_df, valid_df = add_oof_knn_features(train_df, valid_df, y_train)
display(train_df[KNN_OUTPUT_COLS].head())


In [ ]:
TRIP_NUMERIC_COLS = [
    "trip_distance",
    "log_trip_distance",
    "passenger_count",
    "gps_distance_miles",
    "distance_ratio",
    "lat_diff",
    "lon_diff",
    "PU_lon",
    "PU_lat",
    "DO_lon",
    "DO_lat",
    "pickup_airport",
    "dropoff_airport",
    "airport_trip",
    "same_zone",
    "same_borough",
    "interborough_trip",
]
if USE_DURATION_FEATURES:
    TRIP_NUMERIC_COLS += ["duration_min", "log_duration_min", "speed_mph"]

TIME_NUMERIC_COLS = [
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "pickup_day",
    "pickup_weekofyear",
    "is_weekend",
    "is_rush_hour",
    "is_night_tariff",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos",
]

WEATHER_NUMERIC_COLS = [
    "temperature",
    "precipitation",
    "snowfall",
    "has_precipitation",
    "has_snowfall",
    "bad_weather",
    "freezing_weather",
    "precipitation_x_rush_hour",
    "snowfall_x_rush_hour",
    "temperature_x_precipitation",
]

FINAL_NUMERIC_COLS = [
    *TRIP_NUMERIC_COLS,
    *TIME_NUMERIC_COLS,
    *WEATHER_NUMERIC_COLS,
    *target_stat_cols,
    *KNN_OUTPUT_COLS,
]
FINAL_NUMERIC_COLS = [col for col in dict.fromkeys(FINAL_NUMERIC_COLS) if col in train_df.columns]

CATBOOST_CATEGORICAL_COLS = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "PU_Borough",
    "DO_Borough",
    "distance_group",
    "weather_code",
    "route_id",
    "borough_pair",
]
CATBOOST_CATEGORICAL_COLS = [col for col in CATBOOST_CATEGORICAL_COLS if col in train_df.columns]


LINEAR_CATEGORICAL_COLS = [
    "VendorID",
    "RatecodeID",
    "payment_type",
    "store_and_fwd_flag",
    "PU_Borough",
    "DO_Borough",
    "distance_group",
    "weather_code",
]
LINEAR_CATEGORICAL_COLS = [col for col in LINEAR_CATEGORICAL_COLS if col in train_df.columns]
LINEAR_NUMERIC_WITHOUT_KNN = [col for col in FINAL_NUMERIC_COLS if not col.startswith("knn_")]
LINEAR_NUMERIC_WITH_KNN = FINAL_NUMERIC_COLS.copy()

for col in CATBOOST_CATEGORICAL_COLS:
    train_df[col] = train_df[col].astype("string").fillna("unknown").astype(str)
    valid_df[col] = valid_df[col].astype("string").fillna("unknown").astype(str)

MODEL_FEATURE_COLS = FINAL_NUMERIC_COLS + CATBOOST_CATEGORICAL_COLS
X_train_cb = train_df[MODEL_FEATURE_COLS].copy()
X_valid_cb = valid_df[MODEL_FEATURE_COLS].copy()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def build_linear_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return Pipeline(steps=[
        ("preprocess", preprocessor),

        ("model", Ridge(alpha=CFG["linear_alpha"])),
    ])


model_predictions: dict[str, np.ndarray] = {}
trained_models: dict[str, object] = {}

linear_start = time.time()
linear_base_features = LINEAR_NUMERIC_WITHOUT_KNN + LINEAR_CATEGORICAL_COLS
linear_base = build_linear_pipeline(LINEAR_NUMERIC_WITHOUT_KNN, LINEAR_CATEGORICAL_COLS)
linear_base.fit(train_df[linear_base_features], y_train)
linear_base_pred = np.clip(linear_base.predict(valid_df[linear_base_features]), 0, None)
model_predictions["Linear Regression"] = linear_base_pred
trained_models["Linear Regression"] = linear_base

linear_knn_start = time.time()
linear_knn_features = LINEAR_NUMERIC_WITH_KNN + LINEAR_CATEGORICAL_COLS
linear_knn = build_linear_pipeline(LINEAR_NUMERIC_WITH_KNN, LINEAR_CATEGORICAL_COLS)
linear_knn.fit(train_df[linear_knn_features], y_train)
linear_knn_pred = np.clip(linear_knn.predict(valid_df[linear_knn_features]), 0, None)
model_predictions["Linear Regression + KNN"] = linear_knn_pred
trained_models["Linear Regression + KNN"] = linear_knn


In [ ]:
cat_features_indices = [X_train_cb.columns.get_loc(col) for col in CATBOOST_CATEGORICAL_COLS]
train_pool = Pool(X_train_cb, y_train, cat_features=cat_features_indices)
valid_pool = Pool(X_valid_cb, y_valid, cat_features=cat_features_indices)

catboost_params = {
    "loss_function": "RMSE",
    "eval_metric": "MAE",
    "iterations": CFG["cat_iterations"],
    "learning_rate": CFG["cat_learning_rate"],
    "depth": CFG["cat_depth"],
    "l2_leaf_reg": 6.0,
    "random_seed": RANDOM_STATE,
    "od_type": "Iter",
    "od_wait": CFG["early_stopping_rounds"],
    "verbose": 50,
    "allow_writing_files": False,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.85,
    "one_hot_max_size": 32,
}
if CATBOOST_TASK_TYPE == "GPU":
    catboost_params.update({
        "task_type": "GPU",
        "devices": CATBOOST_DEVICES,
    })
else:
    catboost_params.update({
        "task_type": "CPU",
        "thread_count": -1,
    })


cat_start = time.time()
cat_model = CatBoostRegressor(**catboost_params)
cat_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
cat_pred = np.clip(cat_model.predict(valid_pool), 0, None)
model_predictions["CatBoost GPU + KNN"] = cat_pred
trained_models["CatBoost GPU + KNN"] = cat_model


In [ ]:
import joblib


def regression_report(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype="float64")
    y_pred = np.asarray(y_pred, dtype="float64")
    abs_error = np.abs(y_true - y_pred)
    return {
        "MAE_$": mean_absolute_error(y_true, y_pred),
        "RMSE_$": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
        "MedAE_$": median_absolute_error(y_true, y_pred),
        "P90_AE_$": float(np.percentile(abs_error, 90)),
        "P95_AE_$": float(np.percentile(abs_error, 95)),
        "within_$5_%": float((abs_error <= 5).mean() * 100),
        "within_$10_%": float((abs_error <= 10).mean() * 100),
    }


metrics_rows = []
for model_name, pred in model_predictions.items():
    row = {"model": model_name, **regression_report(y_valid, pred)}
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).sort_values("MAE_$").reset_index(drop=True)
display(metrics_df)

predictions_df = pd.DataFrame({"y_true": y_valid.to_numpy(dtype="float32")})
for model_name, pred in model_predictions.items():
    predictions_df[model_name] = pred.astype("float32")
    predictions_df[f"{model_name} abs_error"] = np.abs(predictions_df["y_true"] - predictions_df[model_name]).astype("float32")

importance_df = pd.DataFrame({
    "feature": MODEL_FEATURE_COLS,
    "importance": cat_model.get_feature_importance(valid_pool),
}).sort_values("importance", ascending=False).reset_index(drop=True)

metrics_path = OUTPUT_DIR / "model_comparison_linear_catboost_knn.csv"
pred_path = OUTPUT_DIR / "validation_predictions_sample.csv"
importance_path = OUTPUT_DIR / "catboost_feature_importance.csv"
features_path = OUTPUT_DIR / "catboost_feature_columns.json"
model_path = OUTPUT_DIR / "catboost_gpu_knn_price_model.cbm"
linear_path = OUTPUT_DIR / "linear_regression_baseline.joblib"
linear_knn_path = OUTPUT_DIR / "linear_regression_knn.joblib"

metrics_df.to_csv(metrics_path, index=False)
predictions_df.sample(min(100_000, len(predictions_df)), random_state=RANDOM_STATE).to_csv(pred_path, index=False)
importance_df.to_csv(importance_path, index=False)
with open(features_path, "w", encoding="utf-8") as f:
    json.dump({
        "model_feature_cols": MODEL_FEATURE_COLS,
        "numeric_cols": FINAL_NUMERIC_COLS,
        "catboost_categorical_cols": CATBOOST_CATEGORICAL_COLS,
        "linear_categorical_cols": LINEAR_CATEGORICAL_COLS,
        "target_stat_cols": target_stat_cols,
        "knn_output_cols": KNN_OUTPUT_COLS,
        "run_mode": RUN_MODE,
        "config": CFG,
        "use_duration_features": USE_DURATION_FEATURES,
    }, f, ensure_ascii=False, indent=2)
cat_model.save_model(str(model_path))
joblib.dump(linear_base, linear_path)
joblib.dump(linear_knn, linear_knn_path)


In [ ]:
pio.templates.default = "plotly_dark"

DARK_BG = "#000000"
PAPER_BG = "#020202"
GRID_COLOR = "rgba(255,255,255,0.14)"
WHITE = "#F4F7FB"
MUTED = "#B7C0CC"
RED = "#A90000"
GREEN = "#9BFF3D"
BLUE = "#2F80ED"
CYAN = "#00D1FF"
PURPLE = "#7A5CFF"
MAGENTA = "#D642FF"
ORANGE = "#FFB000"

BAR_PALETTE = [RED, GREEN, BLUE, "#4867D6", "#5B5ED6", "#6D55C8", "#8146B3", "#9340A2", "#A33A91", "#B13280", "#C22B72", "#D32665", "#E42258", "#F11F4C", "#FF2444"]
MODEL_COLORS = {
    "Linear Regression": BLUE,
    "Linear Regression + KNN": PURPLE,
    "CatBoost GPU + KNN": GREEN,
}
MODEL_LABELS = {
    "Linear Regression": "Линейная регрессия",
    "Linear Regression + KNN": "Линейная регрессия + KNN",
    "CatBoost GPU + KNN": "CatBoost GPU + KNN",
}
METRIC_LABELS = {
    "model": "Модель",
    "MAE_$": "Средняя ошибка, $",
    "RMSE_$": "RMSE, $",
    "MedAE_$": "Медианная ошибка, $",
    "P90_AE_$": "90% ошибок не выше, $",
    "P95_AE_$": "95% ошибок не выше, $",
    "R2": "R²",
    "within_$5_%": "Ошибка до $5, %",
    "within_$10_%": "Ошибка до $10, %",
}
FEATURE_LABELS = {
    "te_RatecodeID_mean": "Средняя цена по тарифному коду",
    "te_RatecodeID_count_log": "Частота тарифного кода",
    "te_borough_pair_mean": "Средняя цена по паре районов",
    "te_borough_pair_count_log": "Частота пары районов",
    "knn_mean_total_amount": "Средняя цена похожих поездок",
    "knn_median_total_amount": "Медианная цена похожих поездок",
    "knn_p75_total_amount": "75-й процентиль цены похожих поездок",
    "knn_p25_total_amount": "25-й процентиль цены похожих поездок",
    "knn_min_total_amount": "Минимальная цена похожих поездок",
    "knn_max_total_amount": "Максимальная цена похожих поездок",
    "knn_std_total_amount": "Разброс цен похожих поездок",
    "knn_mean_distance": "Средняя дистанция до похожих поездок",
    "knn_min_distance": "Минимальная дистанция до похожей поездки",
    "is_rush_hour": "Час пик",
    "hour_sin": "Час поездки, sin",
    "hour_cos": "Час поездки, cos",
    "dow_sin": "День недели, sin",
    "dow_cos": "День недели, cos",
    "log_trip_distance": "Логарифм дистанции поездки",
    "distance_group": "Группа дистанции",
    "trip_distance": "Дистанция поездки",
    "log_duration_min": "Логарифм длительности",
    "duration_min": "Длительность поездки",
    "distance_ratio": "Коэффициент извилистости маршрута",
    "DO_Borough": "Район высадки",
    "PU_Borough": "Район посадки",
    "borough_pair": "Пара районов",
    "DO_lon": "Долгота высадки",
    "DO_lat": "Широта высадки",
    "PU_lon": "Долгота посадки",
    "PU_lat": "Широта посадки",
    "speed_mph": "Средняя скорость",
    "gps_distance_miles": "GPS-дистанция",
    "payment_type": "Тип оплаты",
    "RatecodeID": "Тарифный код",
    "VendorID": "Поставщик данных",
    "store_and_fwd_flag": "Флаг сохранения и пересылки",
}


def model_label(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name)


def metric_label(metric_name: str) -> str:
    return METRIC_LABELS.get(metric_name, metric_name)


def feature_label(feature_name: str) -> str:
    return FEATURE_LABELS.get(feature_name, feature_name)
GROUP_COLORS = {
    "KNN local price": GREEN,
    "Локальная цена KNN": GREEN,
    "Route target stats": ORANGE,
    "Маршрутные target-статистики": ORANGE,
    "Trip geometry": BLUE,
    "Геометрия поездки": BLUE,
    "Time": CYAN,
    "Время": CYAN,
    "Weather": PURPLE,
    "Погода": PURPLE,
    "Category": MAGENTA,
    "Категория": MAGENTA,
    "Other": WHITE,
    "Другое": WHITE,
}


def apply_dark_layout(fig: go.Figure, title: str, x_title: str | None = None, y_title: str | None = None, height: int = 650) -> go.Figure:
    fig.update_layout(
        title={
            "text": f"<b>{title}</b>",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.965,
            "font": {"size": 24, "color": WHITE},
        },
        paper_bgcolor=PAPER_BG,
        plot_bgcolor=DARK_BG,
        font={"family": "Arial, sans-serif", "size": 14, "color": WHITE},
        height=height,
        margin={"l": 120, "r": 220, "t": 130, "b": 100},
        legend={
            "orientation": "h",
            "x": 1.0,
            "xanchor": "right",
            "y": 1.02,
            "yanchor": "bottom",
            "bgcolor": "rgba(0,0,0,0)",
            "font": {"color": WHITE, "size": 12},
            "itemsizing": "constant",
        },
        hoverlabel={"bgcolor": "#111111", "font_size": 13, "font_family": "Arial"},
        uniformtext={"minsize": 9, "mode": "hide"},
    )
    fig.update_xaxes(
        title_text=x_title,
        showgrid=True,
        gridcolor=GRID_COLOR,
        zeroline=False,
        color=WHITE,
        automargin=True,
        title_standoff=14,
        title_font={"color": WHITE, "size": 15},
        tickfont={"color": WHITE, "size": 12},
    )
    fig.update_yaxes(
        title_text=y_title,
        showgrid=True,
        gridcolor=GRID_COLOR,
        zeroline=False,
        color=WHITE,
        automargin=True,
        title_standoff=14,
        title_font={"color": WHITE, "size": 15},
        tickfont={"color": WHITE, "size": 12},
    )
    return fig


def save_plotly_figure(fig: go.Figure, name: str, width: int = 1280, height: int = 720) -> None:
    html_path = FIG_DIR / f"{name}.html"
    png_path = FIG_DIR / f"{name}.png"

    if EXPORT_PLOTLY_HTML:
        fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)

    if EXPORT_PLOTLY_PNG and KALEIDO_AVAILABLE:
        try:
            fig.write_image(png_path, width=width, height=height, scale=2)
        except Exception as exc:
            pass
    elif EXPORT_PLOTLY_PNG and not KALEIDO_AVAILABLE:
        pass


def money_text(values) -> list[str]:
    return [f"{v:,.2f}" for v in values]


def pct_text(values) -> list[str]:
    return [f"{v:.1f}%" for v in values]


In [ ]:
plot_base = pd.concat([train_df, valid_df], ignore_index=True)

zone_label_col = None
for candidate in ["PU_Zone", "PU_zone", "PU_zone_name", "pickup_zone", "PULocationID"]:
    if candidate in plot_base.columns:
        zone_label_col = candidate
        break

if zone_label_col is not None:
    top_revenue = (
        plot_base.groupby(zone_label_col, observed=False)[TARGET]
        .sum()
        .sort_values(ascending=False)
        .head(15)
        .sort_values(ascending=True)
        .reset_index()
    )
    bar_colors = BAR_PALETTE[:len(top_revenue)][::-1]
    fig = go.Figure(go.Bar(
        x=top_revenue[TARGET],
        y=top_revenue[zone_label_col].astype(str),
        orientation="h",
        marker={"color": bar_colors},
        text=money_text(top_revenue[TARGET]),
        textposition="inside",
        insidetextanchor="end",
        textfont={"color": WHITE, "size": 11},
        hovertemplate="%{y}<br>Выручка: $%{x:,.2f}<extra></extra>",
    ))
    apply_dark_layout(fig, "Топ-15 районов по общей выручке", "Общая выручка, $", "Район посадки", height=720)
    fig.update_xaxes(tickprefix="$", separatethousands=True)
    save_plotly_figure(fig, "eda_top15_pickup_revenue")
    fig.show()


price_cap = float(plot_base[TARGET].quantile(0.99))
price_plot = plot_base[[TARGET]].copy()
price_plot["total_amount_clipped"] = price_plot[TARGET].clip(upper=price_cap)
fig = px.histogram(
    price_plot,
    x="total_amount_clipped",
    nbins=80,
    color_discrete_sequence=[BLUE],
)
fig.update_traces(marker_line_width=0, opacity=0.92, hovertemplate="Стоимость: $%{x:.2f}<br>Поездок: %{y:,}<extra></extra>")
apply_dark_layout(fig, "Распределение стоимости поездок", "Стоимость поездки, $", "Количество поездок", height=620)
fig.update_xaxes(tickprefix="$")
save_plotly_figure(fig, "eda_total_amount_distribution")
fig.show()


scatter_sample = plot_base.sample(min(CFG["plot_sample_rows"], len(plot_base)), random_state=RANDOM_STATE)
scatter_sample = scatter_sample[(scatter_sample["trip_distance"].between(0, 40)) & (scatter_sample[TARGET].between(0, 180))]
fig = go.Figure(go.Scattergl(
    name="Поездки",
    x=scatter_sample["trip_distance"],
    y=scatter_sample[TARGET],
    mode="markers",
    marker={"size": 4, "opacity": 0.28, "color": GREEN},
    hovertemplate="Дистанция=%{x:.2f} mi<br>Стоимость=$%{y:.2f}<extra></extra>",
))
apply_dark_layout(fig, "Стоимость растёт с дистанцией", "Дистанция, miles", "Стоимость, $", height=620)
fig.update_yaxes(tickprefix="$")
save_plotly_figure(fig, "eda_distance_vs_total_amount")
fig.show()

del plot_base
gc.collect()


In [ ]:
metrics_display = metrics_df.copy()
for col in ["MAE_$", "RMSE_$", "MedAE_$", "P90_AE_$", "P95_AE_$"]:
    metrics_display[col] = metrics_display[col].round(3)
for col in ["R2", "within_$5_%", "within_$10_%"]:
    metrics_display[col] = metrics_display[col].round(3)
metrics_display["model"] = metrics_display["model"].map(model_label)
metrics_table = metrics_display.rename(columns=METRIC_LABELS)


fig = go.Figure(data=[go.Table(
    header={
        "values": [f"<b>{col}</b>" for col in metrics_table.columns],
        "fill_color": "#121224",
        "font": {"color": WHITE, "size": 13},
        "align": "center",
        "height": 34,
    },
    cells={
        "values": [metrics_table[col] for col in metrics_table.columns],
        "fill_color": "#050505",
        "font": {"color": WHITE, "size": 12},
        "align": "center",
        "height": 30,
    },
)])
apply_dark_layout(fig, "Сравнение моделей на проверочной выборке", height=420)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
save_plotly_figure(fig, "model_metrics_table", height=480)
fig.show()


error_metrics = ["MAE_$", "RMSE_$", "MedAE_$", "P90_AE_$", "P95_AE_$"]
error_metric_labels = [metric_label(m) for m in error_metrics]
fig = go.Figure()
for _, row in metrics_df.iterrows():
    model_name = row["model"]
    values = [row[m] for m in error_metrics]
    fig.add_trace(go.Bar(
        x=error_metric_labels,
        y=values,
        name=model_label(model_name),
        marker_color=MODEL_COLORS.get(model_name, BLUE),
        text=money_text(values),
        textposition="outside",
        hovertemplate="%{x}<br>$%{y:.3f}<extra></extra>",
    ))
apply_dark_layout(fig, "Ошибки моделей: чем ниже, тем лучше", "Метрика", "Ошибка, $", height=680)
fig.update_yaxes(tickprefix="$")
fig.update_layout(
    barmode="group",
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": 1.08, "yanchor": "bottom"},
)
save_plotly_figure(fig, "model_error_metrics_comparison")
fig.show()


quality_metrics = ["R2", "within_$5_%", "within_$10_%"]
quality_metric_labels = [metric_label(m) for m in quality_metrics]
fig = go.Figure()
for _, row in metrics_df.iterrows():
    model_name = row["model"]
    values = [row[m] for m in quality_metrics]
    text_values = [f"{values[0]:.3f}", f"{values[1]:.1f}%", f"{values[2]:.1f}%"]
    fig.add_trace(go.Bar(
        x=quality_metric_labels,
        y=values,
        name=model_label(model_name),
        marker_color=MODEL_COLORS.get(model_name, BLUE),
        text=text_values,
        textposition="outside",
        hovertemplate="%{x}<br>%{y:.3f}<extra></extra>",
    ))
apply_dark_layout(fig, "Качество моделей: чем выше, тем лучше", "Метрика", "Значение", height=700)
fig.update_layout(
    barmode="group",
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": 1.08, "yanchor": "bottom"},
)
save_plotly_figure(fig, "model_quality_metrics_comparison")
fig.show()


In [ ]:
plot_pred = predictions_df.sample(min(CFG["plot_sample_rows"], len(predictions_df)), random_state=RANDOM_STATE).copy()
actual_max = float(np.percentile(plot_pred["y_true"], 99))
actual_max = max(actual_max, 20)


fig = go.Figure()
for model_name in model_predictions.keys():
    fig.add_trace(go.Scattergl(
        x=plot_pred["y_true"].clip(0, actual_max),
        y=plot_pred[model_name].clip(0, actual_max),
        mode="markers",
        name=model_label(model_name),
        marker={"size": 4, "opacity": 0.28, "color": MODEL_COLORS.get(model_name, BLUE)},
        hovertemplate="Фактическая=$%{x:.2f}<br>Прогноз=$%{y:.2f}<extra></extra>",
    ))
fig.add_trace(go.Scattergl(
    x=[0, actual_max],
    y=[0, actual_max],
    mode="lines",
    name="Идеальная линия",
    line={"color": RED, "width": 4, "dash": "dash"},
    hoverinfo="skip",
))
apply_dark_layout(fig, "Сравнение фактической и предсказанной стоимости", "Фактическая стоимость, $", "Прогноз, $", height=720)
fig.update_xaxes(tickprefix="$")
fig.update_yaxes(tickprefix="$")
save_plotly_figure(fig, "actual_vs_predicted_models")
fig.show()


fig = go.Figure()
for model_name in model_predictions.keys():
    error_col = f"{model_name} abs_error"
    fig.add_trace(go.Histogram(
        x=predictions_df[error_col].clip(0, 50),
        nbinsx=80,
        name=model_label(model_name),
        marker_color=MODEL_COLORS.get(model_name, BLUE),
        opacity=0.68,
        hovertemplate="Абсолютная ошибка=$%{x:.2f}<br>Количество=%{y:,}<extra></extra>",
    ))
apply_dark_layout(fig, "Распределение абсолютной ошибки", "Абсолютная ошибка, $", "Количество поездок", height=650)
fig.update_layout(barmode="overlay")
fig.update_xaxes(tickprefix="$")
save_plotly_figure(fig, "absolute_error_distribution_models")
fig.show()


final_name = "CatBoost GPU + KNN"
plot_pred["catboost_residual"] = plot_pred["y_true"] - plot_pred[final_name]
fig = go.Figure(go.Scattergl(
    name="Поездки",
    x=plot_pred[final_name].clip(0, actual_max),
    y=plot_pred["catboost_residual"].clip(-40, 40),
    mode="markers",
    marker={"size": 4, "opacity": 0.30, "color": CYAN},
    hovertemplate="Прогноз=$%{x:.2f}<br>Остаток=$%{y:.2f}<extra></extra>",
))
fig.add_shape(
    type="line",
    x0=0,
    x1=actual_max,
    y0=0,
    y1=0,
    line={"color": RED, "width": 4, "dash": "dash"},
    layer="above",
)
apply_dark_layout(fig, "Остатки модели CatBoost + KNN", "Прогноз, $", "Факт − прогноз, $", height=620)
fig.update_xaxes(tickprefix="$")
fig.update_yaxes(tickprefix="$")
save_plotly_figure(fig, "catboost_residuals")
fig.show()


In [ ]:
segment_base = valid_df.reset_index(drop=True).copy()
segment_base["y_true"] = y_valid.to_numpy(dtype="float32")
for model_name, pred in model_predictions.items():
    segment_base[f"{model_name}_abs_error"] = np.abs(segment_base["y_true"] - pred)


fare_bins = [0, 10, 20, 40, 60, 100, 200, 500]
segment_base["fare_band"] = pd.cut(segment_base["y_true"], bins=fare_bins, include_lowest=True).astype(str)
fare_rows = []
for model_name in model_predictions.keys():
    err_col = f"{model_name}_abs_error"
    grouped = segment_base.groupby("fare_band", observed=False)[err_col].mean().reset_index()
    grouped["model"] = model_name
    grouped = grouped.rename(columns={err_col: "MAE_$"})
    fare_rows.append(grouped)
fare_mae_df = pd.concat(fare_rows, ignore_index=True)

fig = go.Figure()
for model_name in model_predictions.keys():
    part = fare_mae_df[fare_mae_df["model"] == model_name]
    fig.add_trace(go.Bar(
        x=part["fare_band"],
        y=part["MAE_$"],
        name=model_label(model_name),
        marker_color=MODEL_COLORS.get(model_name, BLUE),
        text=money_text(part["MAE_$"]),
        textposition="outside",
        hovertemplate="%{x}<br>MAE=$%{y:.2f}<extra></extra>",
    ))
apply_dark_layout(fig, "MAE по диапазонам стоимости", "Фактическая стоимость", "MAE, $", height=650)
fig.update_yaxes(tickprefix="$")
fig.update_layout(barmode="group")
save_plotly_figure(fig, "mae_by_fare_band")
fig.show()


def segment_mae_plot(segment_col: str, title: str, file_name: str, top_n: int = 12):
    if segment_col not in segment_base.columns:
        return
    counts = segment_base[segment_col].astype(str).value_counts().head(top_n)
    keep = counts.index.tolist()
    rows = []
    for model_name in model_predictions.keys():
        err_col = f"{model_name}_abs_error"
        part = segment_base[segment_base[segment_col].astype(str).isin(keep)].copy()
        grouped = part.groupby(segment_col, observed=False)[err_col].mean().reset_index()
        grouped["model"] = model_name
        grouped = grouped.rename(columns={err_col: "MAE_$", segment_col: "segment"})
        rows.append(grouped)
    seg_df = pd.concat(rows, ignore_index=True)
    seg_df["segment"] = seg_df["segment"].astype(str)

    fig = go.Figure()
    for model_name in model_predictions.keys():
        part = seg_df[seg_df["model"] == model_name]
        fig.add_trace(go.Bar(
            x=part["segment"],
            y=part["MAE_$"],
            name=model_label(model_name),
            marker_color=MODEL_COLORS.get(model_name, BLUE),
            text=money_text(part["MAE_$"]),
            textposition="outside",
            hovertemplate="%{x}<br>MAE=$%{y:.2f}<extra></extra>",
        ))
    apply_dark_layout(fig, title, segment_col, "MAE, $", height=650)
    fig.update_yaxes(tickprefix="$")
    fig.update_layout(barmode="group")
    save_plotly_figure(fig, file_name)
    fig.show()


segment_mae_plot("RatecodeID", "MAE по тарифным кодам", "mae_by_ratecode", top_n=10)
segment_mae_plot("PU_Borough", "MAE по borough посадки", "mae_by_pickup_borough", top_n=10)


In [ ]:
def feature_group(feature: str) -> str:
    lower = feature.lower()
    if lower.startswith("knn_"):
        return "Локальная цена KNN"
    if lower.startswith("te_"):
        return "Маршрутные target-статистики"
    if any(token in lower for token in ["distance", "duration", "speed", "airport", "zone", "borough", "lat", "lon"]):
        return "Геометрия поездки"
    if any(token in lower for token in ["pickup_hour", "pickup_day", "month", "dow", "week", "rush", "night", "weekend", "hour_"]):
        return "Время"
    if any(token in lower for token in ["weather", "temperature", "precip", "snow"]):
        return "Погода"
    if feature in CATBOOST_CATEGORICAL_COLS:
        return "Категория"
    return "Другое"


importance_plot = importance_df.copy()
importance_plot["group"] = importance_plot["feature"].apply(feature_group)
importance_plot["feature_ru"] = importance_plot["feature"].apply(feature_label)


top_imp = importance_plot.head(30).sort_values("importance", ascending=True)
fig = px.bar(
    top_imp,
    x="importance",
    y="feature_ru",
    orientation="h",
    color="group",
    color_discrete_map=GROUP_COLORS,
    text="importance",
    custom_data=["feature"],
)
fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    hovertemplate="%{y}<br>Исходное имя: %{customdata[0]}<br>Важность=%{x:.3f}<extra></extra>",
)
apply_dark_layout(fig, "Топ-30 самых важных признаков CatBoost", "Важность", "Признак", height=900)
fig.update_layout(
    margin={"l": 270, "r": 260, "t": 150, "b": 150},
    legend={"orientation": "h", "x": 0.5, "xanchor": "center", "y": -0.16, "yanchor": "top"},
)
fig.update_xaxes(range=[0, float(top_imp["importance"].max()) * 1.16])
save_plotly_figure(fig, "catboost_feature_importance_top30", height=980)
fig.show()


group_imp = importance_plot.groupby("group", as_index=False)["importance"].sum().sort_values("importance", ascending=True)
fig = go.Figure(go.Bar(
    x=group_imp["importance"],
    y=group_imp["group"],
    orientation="h",
    marker_color=[GROUP_COLORS.get(g, WHITE) for g in group_imp["group"]],
    text=[f"{v:.2f}" for v in group_imp["importance"]],
    textposition="outside",
    hovertemplate="%{y}<br>Важность=%{x:.3f}<extra></extra>",
))
apply_dark_layout(fig, "Какие группы признаков влияют на прогноз", "Суммарная важность", "Группа признаков", height=650)
fig.update_layout(margin={"l": 240, "r": 250, "t": 130, "b": 100})
fig.update_xaxes(range=[0, float(group_imp["importance"].max()) * 1.15])
save_plotly_figure(fig, "catboost_feature_importance_by_group")
fig.show()


knn_imp = importance_plot[importance_plot["feature"].str.startswith("knn_")].sort_values("importance", ascending=True)
if not knn_imp.empty:
    fig = go.Figure(go.Bar(
        x=knn_imp["importance"],
        y=knn_imp["feature_ru"],
        orientation="h",
        marker_color=GREEN,
        text=[f"{v:.2f}" for v in knn_imp["importance"]],
        textposition="outside",
        customdata=knn_imp["feature"],
        hovertemplate="%{y}<br>Исходное имя: %{customdata}<br>Важность=%{x:.3f}<extra></extra>",
    ))
    apply_dark_layout(fig, "Вклад KNN-признаков в CatBoost", "Важность", "KNN-признак", height=640)
    fig.update_layout(margin={"l": 330, "r": 260, "t": 130, "b": 100})
    fig.update_xaxes(range=[0, float(knn_imp["importance"].max()) * 1.16])
    save_plotly_figure(fig, "knn_feature_importance")
    fig.show()


In [ ]:
if "knn_mean_total_amount" in valid_df.columns:
    knn_signal = valid_df[[TARGET, "knn_mean_total_amount", "knn_median_total_amount", "knn_std_total_amount"]].copy()
    knn_signal = knn_signal.sample(min(CFG["plot_sample_rows"], len(knn_signal)), random_state=RANDOM_STATE)
    cap = float(np.percentile(knn_signal[TARGET], 99))
    cap = max(cap, 20)

    fig = go.Figure(go.Scattergl(
        name="Поездки",
        x=knn_signal["knn_mean_total_amount"].clip(0, cap),
        y=knn_signal[TARGET].clip(0, cap),
        mode="markers",
        marker={
            "size": 5,
            "opacity": 0.34,
            "color": knn_signal["knn_std_total_amount"].clip(0, 20),
            "colorscale": "Viridis",
            "showscale": True,
            "colorbar": {
                "title": {"text": "Разброс цен KNN", "side": "right", "font": {"color": WHITE}},
                "tickfont": {"color": WHITE},
                "x": 1.04,
                "y": 0.43,
                "len": 0.78,
            },
        },
        hovertemplate="Средняя цена KNN=$%{x:.2f}<br>Фактическая стоимость=$%{y:.2f}<extra></extra>",
    ))
    fig.add_trace(go.Scattergl(
        x=[0, cap],
        y=[0, cap],
        mode="lines",
        name="Идеальная линия",
        line={"color": RED, "width": 4, "dash": "dash"},
        hoverinfo="skip",
    ))
    apply_dark_layout(fig, "Сигнал KNN по локальной цене", "Средняя цена похожих поездок, $", "Фактическая стоимость, $", height=720)
    fig.update_xaxes(tickprefix="$")
    fig.update_yaxes(tickprefix="$")
    save_plotly_figure(fig, "knn_local_price_signal")
    fig.show()


best = metrics_df.iloc[0]
baseline = metrics_df[metrics_df["model"].eq("Linear Regression")].iloc[0]
mae_gain = (baseline["MAE_$"] - best["MAE_$"]) / baseline["MAE_$"] * 100
